In [0]:
# ------------------------------------------------------------
# ---- RESET AUTO LOADER STATE FOR ALL BRONZE STREAMS ----

# KEEP COMMENTED DURING NORMAL RUNS
# Enable ONLY when:
# - Replaying ingestion from scratch
# - Re-uploading same files
# - Source schema changed

# This will delete:
# - Auto Loader schema tracking folders
# - Streaming checkpoints
# ------------------------------------------------------------

schema_base = "/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/schema/"
chk_base    = "/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud/checkpoints/"

bronze_streams = [
    "claims",
    "claim_lines",
    "patients",
    "providers",
    "payments",
    "rules",
    "audit_logs"
]

for s in bronze_streams:
    print(f"Clearing state for: {s}")
    dbutils.fs.rm(schema_base + s, True)
    dbutils.fs.rm(chk_base + "bronze_" + s, True)

print("✔ All Bronze Auto Loader schemas & checkpoints cleared")


In [0]:
# MUST be first line
spark.sql("USE CATALOG health_care_2026")

db = "healthcare_fraud_demo"

# In Unity Catalog use SCHEMA not DATABASE
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

base_path = "/Volumes/health_care_2026/healthcare_fraud_demo/healthcare_fraud"

checkpoint = f"{base_path}/checkpoints"
schema_loc = f"{base_path}/schema"

print(checkpoint)
print(schema_loc)

In [0]:
claims_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header","true")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation", f"{schema_loc}/claims")
    .load(f"{base_path}/HC_2026_claims")
    .withColumn("ingestion_ts", current_timestamp())
)

(claims_stream.writeStream
    .format("delta")
    .option("checkpointLocation", f"{checkpoint}/bronze_claims")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_claims")
)


In [0]:
from pyspark.sql.types import *

claim_line_schema = StructType([
    StructField("claim_id", IntegerType(), True),
    StructField("procedures", ArrayType(
        StructType([
            StructField("code", StringType(), True),
            StructField("charge", DoubleType(), True)
        ])
    ), True)
])

lines_stream = (
    spark.readStream.format("cloudFiles")
    .schema(claim_line_schema)     #ADD THIS
    .option("cloudFiles.format","json")
    .option("multiLine","true")
    .option("cloudFiles.schemaLocation",f"{schema_loc}/claim_lines")
    .load(f"{base_path}/HC_2026_claim_lines")
    .withColumn("ingestion_ts", current_timestamp())
)

(lines_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_claim_lines")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_claim_lines")
)


In [0]:
patients_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("header","true")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",f"{schema_loc}/patients")
    .load(f"{base_path}/HC_2026_patients")
    .withColumn("ingestion_ts", current_timestamp())
)

(patients_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_patients")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_patients")
)

In [0]:
providers_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("header","true")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",f"{schema_loc}/providers")
    .load(f"{base_path}/HC_2026_providers")
    .withColumn("ingestion_ts", current_timestamp())
)

(providers_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_providers")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_providers")
)


In [0]:
payments_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("header","true")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",f"{schema_loc}/payments")
    .load(f"{base_path}/HC_2026_payments")
    .withColumn("ingestion_ts", current_timestamp())
)

(payments_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_payments")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_payments")
)


In [0]:
from pyspark.sql.types import *

rules_schema = StructType([
    StructField("rule_id", StringType(), True),
    StructField("rule_name", StringType(), True),
    StructField("rule_type", StringType(), True),
    StructField("threshold", DoubleType(), True),
    StructField("active_flag", StringType(), True)
])

rules_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{schema_loc}/rules")
    .option("multiLine", "true")   # IMPORTANT for JSON configs
    .schema(rules_schema)          # FIX
    .load(f"{base_path}/HC_2026_rules")
    .withColumn("ingestion_ts", current_timestamp())
)


(rules_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_rules")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_rules")
)


In [0]:
from pyspark.sql.types import *

audit_schema = StructType([
    StructField("audit_id", StringType(), True),
    StructField("claim_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("action", StringType(), True),
    StructField("status", StringType(), True),
    StructField("remarks", StringType(), True)
])

audit_stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{schema_loc}/audit")
    .option("multiLine","true")        # important for JSON logs
    .schema(audit_schema)              # MAIN FIX
    .load(f"{base_path}/HC_2026_audit_logs")
    .withColumn("ingestion_ts", current_timestamp())
)

(audit_stream.writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint}/bronze_audit_logs")
    .trigger(availableNow=True)
    .table(f"{db}.bronze_audit_logs")
)


In [0]:
%sql
SELECT *
FROM healthcare_fraud_demo.bronze_claims
WHERE claim_id = 99999
order by ingestion_ts asc
